# Fetching Defensive Statistics from FBref

This notebook fetches **per-player, per-match defensive statistics** from FBref for Premier League seasons **2019-20 onwards**.

## Data Structure
Each row in the final dataset represents:
- **One player's defensive stats** in **one match** (which corresponds to one gameweek)

## Defensive Stats Include:
- Tackles (total, won, in defensive/middle/attacking third)
- Pressures (total, successful, in each third)
- Blocks (total, shots blocked, passes blocked)
- Interceptions
- Clearances
- Errors leading to shots

In [1]:
import soccerdata as sd
import pandas as pd
import numpy as np
import time
import warnings

# Suppress the FutureWarning about DataFrame concatenation (it's a pandas/soccerdata internal issue)
warnings.filterwarnings("ignore", category=FutureWarning, module="soccerdata")

# Starting from 2019-20 to 2024-25 (6 seasons with actual defensive data)
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2019, 2025)]

print(f"Configured seasons: {seasons}")
print(f"Total: {len(seasons)} seasons")

[01/11/26 14:59:52] INFO     No custom team name replacements found. You can configure these in       ]8;id=976411;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=488631;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\LENOVO\soccerdata\config\teamname_replacements.json.                         

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=929008;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=353038;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_config.py#197\197]8;;\
                             C:\Users\LENOVO\soccerdata\config\league_dict.json.                                   

Configured seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Total: 6 seasons


## Step 1: Initialize FBref Scraper and Get Schedule

First, we initialize the FBref scraper and retrieve the match schedule. This gives us all match IDs which we'll use to fetch individual match statistics.

In [2]:
# Initialize the FBref scraper for all selected seasons
fbref = sd.FBref(leagues="ENG-Premier League", seasons=seasons)

# Read the full schedule (contains match IDs, dates, teams, scores)
schedule = fbref.read_schedule()
schedule_df = schedule.reset_index()

# Display schedule structure
print(f"Schedule shape: {schedule_df.shape}")
print(f"Schedule columns: {schedule_df.columns.tolist()[:15]}...")  # First 15 columns
print(f"\nSample schedule rows:")
print(schedule_df.head(3))

                    INFO     Saving cached data to C:\Users\LENOVO\soccerdata\data\FBref             ]8;id=47507;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=358783;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\_common.py#263\263]8;;\

Schedule shape: (2280, 18)
Schedule columns: ['league', 'season', 'game', 'week', 'day', 'date', 'time', 'home_team', 'home_xg', 'score', 'away_xg', 'away_team', 'attendance', 'venue', 'referee']...

Sample schedule rows:
               league season                                  game  week  day  \
0  ENG-Premier League   1920     2019-08-09 Liverpool-Norwich City     1  Fri   
1  ENG-Premier League   1920  2019-08-10 Bournemouth-Sheffield Utd     1  Sat   
2  ENG-Premier League   1920        2019-08-10 Burnley-Southampton     1  Sat   

        date   time    home_team  home_xg score  away_xg      away_team  \
0 2019-08-09  20:00    Liverpool      1.8   4–1      0.9   Norwich City   
1 2019-08-10  15:00  Bournemouth      1.3   1–1      1.3  Sheffield Utd   
2 2019-08-10  15:00      Burnley      0.9   3–0      1.2    Southampton   

   attendance             venue         referee  \
0       53333           Anfield  Michael Oliver   
1       10714  Vitality Stadium    Kevin Friend   

## Step 2: Extract Match IDs and Build Gameweek Mapping

We need to:
1. Extract all unique match IDs from the schedule
2. Create a mapping from match ID to gameweek number and date (for later enrichment)

In [3]:
# Find the match id column (name varies by soccerdata version)
id_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["match_id", "game_id", "matchid", "gameid"])
    or str(c).lower() in {"id", "match"}
    or str(c).lower().endswith("_id")
    or ("match" in str(c).lower() and "id" in str(c).lower())
    or ("game" in str(c).lower() and "id" in str(c).lower())
]

# Find gameweek/round column
gw_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["week", "round", "matchweek", "gameweek", "matchday"])
]

# Find date column
date_candidates = [
    c for c in schedule_df.columns
    if any(k in str(c).lower() for k in ["date", "time", "kickoff"])
]

# Find season column
season_candidates = [
    c for c in schedule_df.columns
    if "season" in str(c).lower()
]

print(f"Match ID candidates: {id_candidates}")
print(f"Gameweek candidates: {gw_candidates}")
print(f"Date candidates: {date_candidates}")
print(f"Season candidates: {season_candidates}")

# Extract match IDs
if id_candidates:
    match_id_col = id_candidates[0]
    match_ids = (
        schedule_df[match_id_col]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .tolist()
    )
else:
    # Fallback: try index level names
    idx_match_levels = [
        n for n in schedule.index.names
        if n is not None and any(k in str(n).lower() for k in ["match", "game"]) and "id" in str(n).lower()
    ]
    if not idx_match_levels:
        raise KeyError(
            "Could not identify match id column/level in fbref.read_schedule(). "
            f"Columns: {list(schedule_df.columns)[:30]}; index names: {schedule.index.names}"
        )
    level_name = idx_match_levels[0]
    match_id_col = level_name
    match_ids = (
        pd.Index(schedule.index.get_level_values(level_name))
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

# Build a mapping DataFrame: match_id -> season, gameweek, date, home_team, away_team
gw_col = gw_candidates[0] if gw_candidates else None
date_col = date_candidates[0] if date_candidates else None
season_col = season_candidates[0] if season_candidates else None

# Build the mapping
mapping_cols = [match_id_col]
if season_col: mapping_cols.append(season_col)
if gw_col: mapping_cols.append(gw_col)
if date_col: mapping_cols.append(date_col)

# Add team columns if available
team_cols = [c for c in schedule_df.columns if "home" in str(c).lower() or "away" in str(c).lower()]
for tc in team_cols[:4]:  # Limit to first 4 team-related columns
    if tc not in mapping_cols:
        mapping_cols.append(tc)

match_mapping = schedule_df[mapping_cols].drop_duplicates()
match_mapping[match_id_col] = match_mapping[match_id_col].astype(str).str.strip()

print(f"\nMatches found in schedule: {len(match_ids):,}")
print(f"Match mapping columns: {match_mapping.columns.tolist()}")
print(f"\nSample mapping:")
print(match_mapping.head(5))

Match ID candidates: ['game_id']
Gameweek candidates: ['week']
Date candidates: ['date', 'time']
Season candidates: ['season']

Matches found in schedule: 2,280
Match mapping columns: ['game_id', 'season', 'week', 'date', 'home_team', 'home_xg', 'away_xg', 'away_team']

Sample mapping:
    game_id season  week       date       home_team  home_xg  away_xg  \
0  928467bd   1920     1 2019-08-09       Liverpool      1.8      0.9   
1  d402cacd   1920     1 2019-08-10     Bournemouth      1.3      1.3   
2  34b99058   1920     1 2019-08-10         Burnley      0.9      1.2   
3  a802f51e   1920     1 2019-08-10  Crystal Palace      0.9      1.1   
4  404ee5d3   1920     1 2019-08-10       Tottenham      2.4      0.7   

       away_team  
0   Norwich City  
1  Sheffield Utd  
2    Southampton  
3        Everton  
4    Aston Villa  


## Step 3: Fetch Defensive Stats for All Matches

This is the main data collection loop. We fetch defensive statistics for each match individually to:
1. Handle failures gracefully (one failed match won't crash the entire process)
2. Track progress and failures for debugging
3. Add gentle pacing to avoid being blocked by FBref

In [4]:
dfs = []           # Successfully fetched dataframes
failed = []        # Failed match IDs with error info
empty_matches = [] # Matches with no stats available

print(f"Starting to fetch defensive stats for {len(match_ids):,} matches...")
print("This may take a while. Progress updates every 100 matches.\n")

start_time = time.time()

for i, match_id in enumerate(match_ids, start=1):
    try:
        # Fetch defensive stats for this match
        df = fbref.read_player_match_stats(
            stat_type="defense",
            match_id=match_id,
            force_cache=True,  # Use cached data if available
        )
        
        # Check if we got valid data
        if df is not None and not df.empty:
            # Add match_id to the dataframe for later mapping
            df_reset = df.reset_index()
            df_reset['match_id'] = match_id
            dfs.append(df_reset)
        else:
            # Match exists but has no defensive stats (common for older seasons)
            empty_matches.append(match_id)
            
    except Exception as e:
        error_msg = str(e)
        # Only log actual errors, not "no stats" messages
        if "No stats found" not in error_msg:
            failed.append((match_id, type(e).__name__, error_msg[:100]))
        else:
            empty_matches.append(match_id)
    
    # Progress logging every 100 matches
    if i % 100 == 0:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        eta = (len(match_ids) - i) / rate if rate > 0 else 0
        print(f"Progress: {i:,}/{len(match_ids):,} matches ({i/len(match_ids)*100:.1f}%)")
        print(f"  ✓ Success: {len(dfs):,} | ⚠ Empty: {len(empty_matches):,} | ✗ Failed: {len(failed):,}")
        print(f"  Time: {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining\n")
        time.sleep(0.3)  # Gentle pacing to avoid rate limiting

# Final summary
total_time = time.time() - start_time
print("="*60)
print("FETCH COMPLETE")
print("="*60)
print(f"Total matches processed: {len(match_ids):,}")
print(f"  ✓ Successfully fetched: {len(dfs):,}")
print(f"  ⚠ Empty (no stats available): {len(empty_matches):,}")
print(f"  ✗ Failed with errors: {len(failed):,}")
print(f"Total time: {total_time/60:.1f} minutes")

Starting to fetch defensive stats for 2,280 matches...
This may take a while. Progress updates every 100 matches.



[01/11/26 14:59:57] INFO     [1/1] Retrieving game with id=928467bd                                    ]8;id=582936;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=918288;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 14:59:58] INFO     [1/1] Retrieving game with id=d402cacd                                    ]8;id=230049;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=115111;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 14:59:59] INFO     [1/1] Retrieving game with id=34b99058                                    ]8;id=172298;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=614758;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 15:00:00] INFO     [1/1] Retrieving game with id=a802f51e                                    ]8;id=249213;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=461209;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

...

[01/11/26 17:44:14] INFO     [1/1] Retrieving game with id=464cbad6                                    ]8;id=897219;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=916413;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 17:44:21] INFO     [1/1] Retrieving game with id=01d155b4                                    ]8;id=540205;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=138634;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 17:44:28] INFO     [1/1] Retrieving game with id=e4bb1c35                                    ]8;id=367738;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=552940;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

[01/11/26 17:44:35] INFO     [1/1] Retrieving game with id=812ef8ad                                    ]8;id=532761;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py\fbref.py]8;;\:]8;id=130195;file://c:\Users\LENOVO\anaconda3\envs\myenv\Lib\site-packages\soccerdata\fbref.py#815\815]8;;\

FETCH COMPLETE
Total matches processed: 2,280
  ✓ Successfully fetched: 2,280
  ⚠ Empty (no stats available): 0
  ✗ Failed with errors: 0
Total time: 164.8 minutes


In [8]:
# print sample of data to see how to merge with mapping later
if dfs:
    sample_df = dfs[0]
    print("\nSample fetched defensive stats dataframe:")
    print(sample_df.head(3))


Sample fetched defensive stats dataframe:
               league season                               game       team  \
                                                                             
0  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
1  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
2  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   

             player jersey_number nation pos     age min  ... Challenges  \
                                                          ...       Lost   
0            Adrián            13    ESP  GK  32-218  52  ...          0   
1           Alisson             1    BRA  GK  26-311  38  ...          0   
2  Andrew Robertson            26    SCO  LB  25-151  90  ...          0   

  Blocks         Int Tkl+Int Clr Err   game_id  match_id  
  Blocks Sh Pass                                          
0      0  0    0   0       0   0   0  928467bd  928467bd  
1      0

In [ ]:
'''Sample fetched defensive stats dataframe:
               league season                               game       team  \
                                                                             
0  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
1  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   
2  ENG-Premier League   1920  2019-08-09 Liverpool-Norwich City  Liverpool   

             player jersey_number nation pos     age min  ... Challenges  \
                                                          ...       Lost   
0            Adrián            13    ESP  GK  32-218  52  ...          0   
1           Alisson             1    BRA  GK  26-311  38  ...          0   
2  Andrew Robertson            26    SCO  LB  25-151  90  ...          0   

  Blocks         Int Tkl+Int Clr Err   game_id  match_id  
  Blocks Sh Pass                                          
0      0  0    0   0       0   0   0  928467bd  928467bd  
1      0  0    0   0       0   0   1  928467bd  928467bd  
2      2  0    2   1       2   1   0  928467bd  928467bd  
'''
# this was the output
# now we merge:
# converting the dfs list into a single dataframe
if dfs:
    defensive_stats_df = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined defensive stats dataframe shape: {defensive_stats_df.shape}")
    # Merging with match mapping to get season, gameweek, date, teams
    


Combined defensive stats dataframe shape: (65788, 28)


In [14]:
defensive_stats_df.to_csv("defensive_stats_raw.csv", index=False)

## Step 4: Combine and Enrich Data

Now we:
1. Concatenate all fetched dataframes
2. Merge with the match mapping to add season, gameweek, and date information
3. Clean up column names and organize the data

In [5]:
if not dfs:
    raise RuntimeError(
        "No match stats were fetched. This could be due to:\n"
        "1. FBref blocking requests - try again later or from a different network\n"
        "2. Network connectivity issues\n"
        "3. soccerdata version incompatibility"
    )

# Concatenate all fetched dataframes
print("Combining all fetched data...")
player_stats_all = pd.concat(dfs, axis=0, ignore_index=True)

print(f"Combined dataframe shape: {player_stats_all.shape}")
print(f"Columns: {player_stats_all.columns.tolist()[:20]}...")

# Merge with match mapping to add season, gameweek, date info
print("\nEnriching with season/gameweek information...")
player_df = player_stats_all.merge(
    match_mapping,
    left_on='match_id',
    right_on=match_id_col,
    how='left'
)

# If match_id_col is different from 'match_id', drop the duplicate
if match_id_col != 'match_id' and match_id_col in player_df.columns:
    player_df = player_df.drop(columns=[match_id_col])

print(f"Enriched dataframe shape: {player_df.shape}")
print(f"Final columns ({len(player_df.columns)}): {player_df.columns.tolist()}")

Combining all fetched data...
Combined dataframe shape: (65788, 28)
Columns: [('league', ''), ('season', ''), ('game', ''), ('team', ''), ('player', ''), ('jersey_number', ''), ('nation', ''), ('pos', ''), ('age', ''), ('min', ''), ('Tackles', 'Tkl'), ('Tackles', 'TklW'), ('Tackles', 'Def 3rd'), ('Tackles', 'Mid 3rd'), ('Tackles', 'Att 3rd'), ('Challenges', 'Tkl'), ('Challenges', 'Att'), ('Challenges', 'Tkl%'), ('Challenges', 'Lost'), ('Blocks', 'Blocks')]...

Enriching with season/gameweek information...


MergeError: Not allowed to merge between different levels. (2 levels on the left, 1 on the right)

## Step 5: Data Summary and Quality Check

Let's examine the data quality and distribution across seasons.

In [ ]:
# Find player name column
name_candidates = [c for c in ["player", "Player", "player_name", "name"] if c in player_df.columns]
if not name_candidates:
    name_candidates = [
        c for c in player_df.columns
        if "player" in str(c).lower() or str(c).lower() == "name"
    ]
name_col = name_candidates[0] if name_candidates else None

print("="*60)
print("DATA SUMMARY")
print("="*60)
print(f"Total rows (player-match records): {len(player_df):,}")

if name_col:
    unique_players = player_df[name_col].dropna().astype(str).str.strip().loc[lambda s: s.ne("")].nunique()
    print(f"Unique players: {unique_players:,}")

# Check season distribution if season column exists
if season_col and season_col in player_df.columns:
    print(f"\nRecords per season:")
    season_counts = player_df[season_col].value_counts().sort_index()
    for season, count in season_counts.items():
        print(f"  {season}: {count:,} records")

# Check gameweek distribution if available
if gw_col and gw_col in player_df.columns:
    print(f"\nGameweek range: {player_df[gw_col].min()} to {player_df[gw_col].max()}")

# Show sample data
print("\n" + "="*60)
print("SAMPLE DATA (first 5 rows)")
print("="*60)
display_cols = [c for c in player_df.columns if not str(c).startswith('_')][:12]
print(player_df[display_cols].head())

## Step 6: Build Player List and Save Data

Finally, we:
1. Create a list of unique players across all seasons
2. Save the full defensive stats dataset
3. Save the player list
4. Optionally save failure logs for debugging

In [ ]:
if name_col:
    # Build unique player list
    player_list = (
        player_df[name_col]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .drop_duplicates()
        .sort_values(kind="stable")
        .reset_index(drop=True)
        .to_frame(name="player")
    )
    
    print(f"Unique players extracted: {len(player_list):,}")
    print("\nFirst 30 players:")
    print(player_list.head(30))
    
    # Save player list
    player_list.to_csv("fbref_epl_defense_players_2018-19_to_2024-25.csv", index=False)
    print(f"\n✓ Player list saved to: fbref_epl_defense_players_2018-19_to_2024-25.csv")

# Save the full defensive stats dataset
output_filename = "fbref_epl_defensive_stats_2018-19_to_2024-25.csv"
player_df.to_csv(output_filename, index=False)
print(f"✓ Full defensive stats saved to: {output_filename}")

# Save failures for debugging (if any)
if failed:
    failed_df = pd.DataFrame(failed, columns=["match_id", "error_type", "error_message"])
    failed_df.to_csv("fbref_failed_matches_defense_2018-19_to_2024-25.csv", index=False)
    print(f"⚠ Failed matches log saved to: fbref_failed_matches_defense_2018-19_to_2024-25.csv")

# Save empty matches list (matches with no stats available)
if empty_matches:
    empty_df = pd.DataFrame({"match_id": empty_matches})
    empty_df.to_csv("fbref_empty_matches_defense_2018-19_to_2024-25.csv", index=False)
    print(f"⚠ Empty matches log saved ({len(empty_matches)} matches without defensive stats)")